# 02 — Train DenseNet121 (5-Fold Cross-Validation)

This notebook trains DenseNet121 using a two-phase strategy:
1. **Head Training** — Base frozen, new classification head trained.
2. **Fine-Tuning** — Full model unfrozen, trained with a very low learning rate.


## 1. Vérification GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
if not gpus:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')
else:
    for gpu in gpus:
        print(f'  {gpu.name}')

## 2. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cellule unique de configuration utilisateur

In [ ]:
REPO_URL = (
    "https://github.com/MyElhadri/"
    "histology-ai-classification.git"
)

REPO_BRANCH = "main"

DRIVE_DATASET = (
    "/content/drive/MyDrive/"
    "histology-ai-classification/data/"
    "nuinsseg_human_22_original"
)

DRIVE_RESULTS = (
    "/content/drive/MyDrive/histology-results"
)

RUN_MODE = "selected_folds"

TRAIN_SINGLE_FOLD = 0

SELECTED_FOLDS = [0, 1]

CONFIRM_FULL_TRAINING = False

## 4. Clonage du dépôt main dans /content

In [ ]:
import os
import shutil

REPO_DIR = '/content/histology-ai-classification'
if os.path.exists(REPO_DIR):
    print(f'Removing old repo at {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

print(f'Cloning {REPO_URL} (branch {REPO_BRANCH}) to {REPO_DIR}')
!git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 5. Installation des dépendances

In [ ]:
!pip install -q -r requirements-colab.txt

## 6. Copie du dataset depuis Drive vers /content

In [ ]:
import os
import shutil

DEST_DATASET = '/content/histology-ai-classification/data/raw/nuinsseg_human_22_original'

if not os.path.exists(DRIVE_DATASET):
    raise FileNotFoundError(f'Source Drive dataset missing: {DRIVE_DATASET}')

print(f'Copying dataset from {DRIVE_DATASET} to {DEST_DATASET}...')
if os.path.exists(DEST_DATASET):
    shutil.rmtree(DEST_DATASET)
shutil.copytree(DRIVE_DATASET, DEST_DATASET)
print('Copy complete.')

## 7. Vérification 22 classes et 432 images

In [ ]:
from pathlib import Path

ds_path = Path(DEST_DATASET)
folders = [f for f in ds_path.iterdir() if f.is_dir()]
if len(folders) != 22:
    raise ValueError(f'Expected 22 folders, found {len(folders)}')

image_count = sum(1 for f in ds_path.rglob('*') if f.is_file())
if image_count != 432:
    raise ValueError(f'Expected 432 images, found {image_count}')

print(f'Dataset verified: 22 folders, {image_count} images.')

## 8. Vérification du fichier des folds

In [ ]:
import pandas as pd
import os

folds_file = '/content/histology-ai-classification/data/manifests/densenet121_folds.csv'
if not os.path.exists(folds_file):
    raise FileNotFoundError(f'Folds file missing: {folds_file}')

df = pd.read_csv(folds_file)
if len(df) != 432:
    raise ValueError(f'Expected 432 rows in folds file, found {len(df)}')

expected_folds = {0, 1, 2, 3, 4}
actual_folds = set(df['fold'].unique())
if expected_folds != actual_folds:
    raise ValueError(f'Expected folds {expected_folds}, found {actual_folds}')

for idx, row in df.iterrows():
    img_path = Path(row['image_path'])
    full_path = Path('/content/histology-ai-classification') / img_path
    if not full_path.exists():
        raise FileNotFoundError(f'Missing image path in folds: {full_path}')

print('Folds file verified: 432 rows, folds 0-4 present, all image paths exist.')

## 9. Chargement du YAML

In [ ]:
from src.utils.config import load_yaml
CONFIG_PATH = 'configs/experiments/densenet121_exp_a_rich_augmentation.yaml'
config = load_yaml(CONFIG_PATH)

print('Dataset root :', config['data']['dataset_root'])
print('Folds path   :', config['data']['folds_path'])
print('Num classes  :', config['data']['num_classes'])
print('Num folds    :', config['validation']['num_folds'])

## 10. Choix du mode d’exécution & Entraînement

In [ ]:
import copy
from pathlib import Path
from src.training.train import train_fold, run_cross_validation
import os

os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Run mode: {RUN_MODE}')

if RUN_MODE == 'smoke_test':
    print('Running Smoke Test (Fold 0, Batch 8, 1 epoch, class_weights enabled, separate folder)...')
    smoke_config = copy.deepcopy(config)
    smoke_config['training']['head_epochs'] = 1
    smoke_config['training']['fine_tuning_epochs'] = 0
    smoke_config['training']['batch_size'] = 8
    smoke_config['training']['use_class_weights'] = True
    out_dir = Path(DRIVE_RESULTS) / 'smoke_test'
    train_fold(smoke_config, fold=0, output_dir=out_dir)

elif RUN_MODE == 'single_fold':
    print(f'Training single fold: {TRAIN_SINGLE_FOLD} using YAML hyperparameters')
    out_dir = Path(DRIVE_RESULTS)
    train_fold(config, fold=TRAIN_SINGLE_FOLD, output_dir=out_dir)

elif RUN_MODE == 'all_folds':
    if not CONFIRM_FULL_TRAINING:
        raise ValueError('CONFIRM_FULL_TRAINING must be True to run all_folds mode.')
    print('Running full 5-fold cross-validation...')
    run_cross_validation(CONFIG_PATH, DRIVE_RESULTS)

elif RUN_MODE == 'selected_folds':
    import gc, shutil, json
    import tensorflow as tf
    print(f'Running Selected Folds mode: {SELECTED_FOLDS}')
    out_dir = Path(DRIVE_RESULTS) / 'densenet121-exp-a-rich-augmentation-screening'
    os.makedirs(out_dir, exist_ok=True)
    shutil.copy(CONFIG_PATH, out_dir / 'used_config.yaml')
    
    selected_metrics = {}
    for fold in SELECTED_FOLDS:
        print(f'\n--- Training Fold {fold} ---')
        metrics = train_fold(config, fold=fold, output_dir=out_dir)
        selected_metrics[f'fold_{fold}'] = metrics
        tf.keras.backend.clear_session()
        gc.collect()
    
    avg_acc = sum(m['accuracy'] for m in selected_metrics.values()) / len(selected_metrics)
    avg_macro = sum(m['macro_f1'] for m in selected_metrics.values()) / len(selected_metrics)
    avg_weighted = sum(m['weighted_f1'] for m in selected_metrics.values()) / len(selected_metrics)
    
    summary = {
        'folds_executed': SELECTED_FOLDS,
        'metrics': selected_metrics,
        'average': {
            'accuracy': avg_acc,
            'macro_f1': avg_macro,
            'weighted_f1': avg_weighted
        }
    }
    metrics_dir = out_dir / 'reports' / 'densenet121' / 'metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    with open(metrics_dir / 'selected_folds_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

## 11. Affichage des résultats

In [ ]:
import json
from pathlib import Path

metrics_dir = Path(DRIVE_RESULTS) / 'reports' / 'densenet121' / 'metrics'

if not metrics_dir.exists():
    print(f'Metrics directory not found: {metrics_dir}')
else:
    summary_path = metrics_dir / 'cross_validation_summary.json'
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print('Average Accuracy:', summary['average']['accuracy'])
        print('Average Macro F1:', summary['average']['macro_f1'])
    else:
        print('Summary not available (run mode was not all_folds or training incomplete).')
        for fold_file in sorted(metrics_dir.glob('fold_*.json')):
            if 'class_weights' in fold_file.name:
                continue
            with open(fold_file) as f:
                m = json.load(f)
            print(f'{fold_file.stem}: accuracy={m.get("accuracy", "N/A")}, macro_f1={m.get("macro_f1", "N/A")}')
